In [ ]:
import torch
import torch.nn as nn

## Simple self-attention without trainable weights



In [ ]:
inputs = torch.tensor(
  [
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # Journey
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]  # step
  ]
)
inputs.shape, inputs.dtype # [context_length, n_embd]
# can assume we are just working in non-batched scenario atm

(torch.Size([6, 3]), torch.float32)

In [ ]:
# Calculating attn weights w.r.t second token

query = inputs[1]
#print(query, query.shape)
attn_weights_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  #print(x_i, x_i.shape)
  attn_weights_2[i] = torch.dot(x_i, query)

attn_weights_2, attn_weights_2.shape

(tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]), torch.Size([6]))

In [ ]:
# Normalized attn scores w.r.t second token

attn_weights_2 = torch.softmax(attn_weights_2, dim=-1)
attn_weights_2, attn_weights_2.sum()

(tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]), tensor(1.))

In [ ]:
# Calculating context vector w.r.t second token

context_vec_2 = torch.zeros(inputs.shape[1])
for i, alpha_i in enumerate(attn_weights_2):
  context_vec_2 += inputs[i] * alpha_i

context_vec_2, context_vec_2.shape # weighted sum multipled by attn score

(tensor([0.4419, 0.6515, 0.5683]), torch.Size([3]))

In [ ]:
# Computing attention weights for all input tokens

attn_weights = inputs @ inputs.T
attn_weights = torch.softmax(attn_weights, dim=-1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [ ]:
# Computing context vectors for all tokens
context_vec = attn_weights @ inputs
context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## Implementing self-attention with trainable weights

In [ ]:
class SelfAttentionV1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out))
    self.W_key = nn.Parameter(torch.rand(d_in, d_out))
    self.W_value = nn.Parameter(torch.rand(d_in, d_out))

  def forward(self, x): # x -> [T, d_in]
    queries = x @ self.W_query # [T, d_out] ; T -> context length
    keys = x @ self.W_key # [T, d_out]
    values = x @ self.W_value # [T, d_out]

    d_out = keys.shape[-1]
    attn_scores = queries @ keys.T # [T, T]
    attn_weights = torch.softmax((attn_scores/d_out**0.5), dim=-1)
    context_vec = attn_weights @ values
    return context_vec

In [ ]:
torch.manual_seed(123)
d_in = inputs.shape[1] # 3
d_out = 2
sa_v1 = SelfAttentionV1(d_in, d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [ ]:
# Using nn.Linear instead of nn.Parameter
# Why ? better initialization

class SelfAttentionV2(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x): # x -> [T, d_in]
    queries = self.W_query(x) # [T, d_out] ; T -> context length
    keys = self.W_key(x) # [T, d_out]
    values = self.W_value(x) # [T, d_out]

    d_out = keys.shape[-1]
    attn_scores = queries @ keys.T # [T, T]
    attn_weights = torch.softmax((attn_scores/d_out**0.5), dim=-1)
    context_vec = attn_weights @ values
    return context_vec

In [ ]:
torch.manual_seed(789)
d_in = inputs.shape[1]
d_out = 2
sa_v2 = SelfAttentionV2(d_in, d_out)
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

## EXERCISE 3.1

In [ ]:
sa_v2.W_query.weight.shape, sa_v1.W_query.shape

(torch.Size([2, 3]), torch.Size([3, 2]))

In [ ]:
sa_v2.W_query.weight.dtype, sa_v1.W_query.dtype

(torch.float32, torch.float32)

In [ ]:
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T)

print(torch.allclose(sa_v1(inputs), sa_v2(inputs)))
sa_v1(inputs)


True


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

## Causal Attention

In [ ]:
queries = sa_v2.W_query(inputs) # [T, d_out] ; T -> context length
keys = sa_v2.W_key(inputs) # [T, d_out]
values = sa_v2.W_value(inputs) # [T, d_out]
d_out = keys.shape[-1]
attn_scores = queries @ keys.T # [T, T]
attn_weights = torch.softmax((attn_scores/d_out**0.5), dim=-1)
print(attn_weights.shape)
print(attn_weights)

context_length = attn_weights.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

masked_simple = mask_simple * attn_weights
print(masked_simple)

masked_simple_norm = masked_simple/masked_simple.sum(dim=1, keepdim=True)
print(masked_simple_norm)

torch.Size([6, 6])
tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)
tensor([[1.0

In [ ]:
# More efficient way
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [ ]:
attn_weights = torch.softmax(masked/d_out**0.5, dim=1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
# Dropout
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
dropout(attn_weights) # scales by (1/0.5)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)

In [ ]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x): # x -> [B, T, d_in]
    B, T, d_in = x.shape
    queries = self.W_query(x) # [B, T, d_out] ; T -> context length
    keys = self.W_key(x) # [B, T, d_out]
    values = self.W_value(x) # [B, T, d_out]

    d_out = keys.shape[-1]
    attn_scores = queries @ keys.transpose(1,2) # [B, T, T]
    attn_scores.masked_fill_(self.mask.bool()[:T, :T], -torch.inf)
    attn_weights = torch.softmax((attn_scores/d_out**0.5), dim=-1)
    # add dropout post everything
    attn_weights = self.dropout(attn_weights)

    context_vecs = attn_weights @ values # [B, T, T] @ [B, T, d_out]
    return context_vecs

In [ ]:
batch = torch.stack([inputs, inputs], dim=0)
batch.shape

torch.Size([2, 6, 3])

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
context_vecs.shape, context_vecs

(torch.Size([2, 6, 2]),
 tensor([[[-0.4519,  0.2216],
          [-0.5874,  0.0058],
          [-0.6300, -0.0632],
          [-0.5675, -0.0843],
          [-0.5526, -0.0981],
          [-0.5299, -0.1081]],
 
         [[-0.4519,  0.2216],
          [-0.5874,  0.0058],
          [-0.6300, -0.0632],
          [-0.5675, -0.0843],
          [-0.5526, -0.0981],
          [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>))

## Multi-head attention

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList([ # IMP. For registering module, params etc
        CausalAttention(
            d_in, d_out, context_length, dropout, qkv_bias
        )
        for _ in range(num_heads)
    ])

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    return out

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mhaw = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mhaw(batch)
context_vecs.shape, context_vecs

(torch.Size([2, 6, 4]),
 tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
          [-0.5874,  0.0058,  0.5891,  0.3257],
          [-0.6300, -0.0632,  0.6202,  0.3860],
          [-0.5675, -0.0843,  0.5478,  0.3589],
          [-0.5526, -0.0981,  0.5321,  0.3428],
          [-0.5299, -0.1081,  0.5077,  0.3493]],
 
         [[-0.4519,  0.2216,  0.4772,  0.1063],
          [-0.5874,  0.0058,  0.5891,  0.3257],
          [-0.6300, -0.0632,  0.6202,  0.3860],
          [-0.5675, -0.0843,  0.5478,  0.3589],
          [-0.5526, -0.0981,  0.5321,  0.3428],
          [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>))

## Efficient Multi-head attention

Parallel computation instead of sequential

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert(d_out % num_heads == 0)

    self.head_dim = d_out // num_heads
    self.num_heads = num_heads

    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )
    self.projection = nn.Linear(d_out, d_out)

  def forward(self, x):
    # x -> [B, T, d_in]
    B, T, d_in = x.shape

    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    queries = queries.view(B, T, self.num_heads, self.head_dim) # [B, T, NH, HD]
    queries = queries.transpose(1, 2) # [B, NH, T, HD]
    keys = keys.view(B, T, self.num_heads, self.head_dim) # [B, T, NH, HD]
    keys = keys.transpose(1, 2) # [B, NH, T, HD]
    values = values.view(B, T, self.num_heads, self.head_dim)
    values = values.transpose(1, 2) # [B, NH, T, HD]

    attn_scores = queries @ keys.transpose(2, 3) # [B, NH, T, T]
    attn_scores.masked_fill_(mask.bool(), -torch.inf)
    attn_weights = torch.softmax(attn_scores/self.head_dim**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vecs = attn_weights @ values # [B, NH, T, HD]
    context_vecs = context_vecs.transpose(1,2).reshape(B, T, -1)
    # Reshape instead of view because view needs contiguous
    # Alternative context_vecs.contiguous().view(B, T, -1)
    #context_vecs = self.projection(context_vecs) # [B, T, d_out]
    return context_vecs

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
num_heads = 2
d_in, d_out = 3, 2*num_heads
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)
context_vecs = mha(batch)
context_vecs.shape, context_vecs

(torch.Size([2, 6, 4]),
 tensor([[[-0.3132, -0.2272,  0.4772,  0.1063],
          [-0.2308,  0.0329,  0.5764,  0.3007],
          [-0.2059,  0.1190,  0.6097,  0.3654],
          [-0.1642,  0.1340,  0.5431,  0.3503],
          [-0.1689,  0.1794,  0.5296,  0.3389],
          [-0.1407,  0.1699,  0.5040,  0.3403]],
 
         [[-0.3132, -0.2272,  0.4772,  0.1063],
          [-0.2308,  0.0329,  0.5764,  0.3007],
          [-0.2059,  0.1190,  0.6097,  0.3654],
          [-0.1642,  0.1340,  0.5431,  0.3503],
          [-0.1689,  0.1794,  0.5296,  0.3389],
          [-0.1407,  0.1699,  0.5040,  0.3403]]], grad_fn=<UnsafeViewBackward0>))